In [ ]:
import panel as pn
import pandas as pd
from config.database import SessionLocal
from models import Cuidador

# Habilita a extensão com design moderno e garante preenchimento de tela
pn.extension('tabulator', design='material', sizing_mode="stretch_width")

# Injetando CSS Customizado para o visual "Premium"
custom_css = """
.premium-btn {
    border-radius: 6px !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    transition: all 0.2s ease-in-out !important;
}
.premium-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(0,0,0,0.15) !important;
}
.premium-card-title {
    font-size: 1.1rem !important;
    color: #1e293b !important;
    font-weight: 700 !important;
}
"""
pn.config.raw_css.append(custom_css)

# Feedbacks visuais
status_alert = pn.pane.Alert("", alert_type="light", visible=False, sizing_mode="stretch_width")

# Componentes de Entrada
txt_cpf = pn.widgets.TextInput(name="CPF Cuidador", placeholder="Ex: 222.222.222-22")
txt_nome = pn.widgets.TextInput(name="Nome Completo", placeholder="Ex: Maria Souza")
txt_dias = pn.widgets.TextInput(name="Dias Disponíveis", placeholder="Ex: Seg a Sex")
txt_horas = pn.widgets.TextInput(name="Horários", placeholder="Ex: 08h-18h")

# Botões estilizados
btn_salvar = pn.widgets.Button(name="Salvar / Atualizar", button_type="primary", icon="device-floppy", css_classes=['premium-btn'])
btn_excluir = pn.widgets.Button(name="Excluir Selecionado", button_type="danger", icon="trash", css_classes=['premium-btn'])
btn_limpar = pn.widgets.Button(name="Limpar Formulário", button_type="light", icon="eraser", css_classes=['premium-btn'])

# Tabela com design moderno e paginação
tabela_dados = pn.widgets.Tabulator(
    pd.DataFrame(),
    height=320,
    layout='fit_columns',
    selectable='checkbox',
    theme='fast',
    pagination='remote',
    page_size=7
)

def carregar_dados():
    db = SessionLocal()
    try:
        query = db.query(Cuidador).all()
        dados = [{"CPF": c.cpf_cuidador, "Nome": c.nome_completo, "Dias": c.dias_disponiveis, "Horas": c.horario_disponivel} for c in query]
        df = pd.DataFrame(dados)
        tabela_dados.value = df
    finally:
        db.close()

@pn.depends(tabela_dados.param.selection, watch=True)
def preencher_formulario(selection):
    if not selection:
        limpar_campos_sem_limpar_selecao()
        return
    linha_selecionada = tabela_dados.value.iloc[selection[0]]
    txt_cpf.value = str(linha_selecionada['CPF'])
    txt_nome.value = str(linha_selecionada['Nome'])
    txt_dias.value = str(linha_selecionada['Dias']) if pd.notna(linha_selecionada['Dias']) else ""
    txt_horas.value = str(linha_selecionada['Horas']) if pd.notna(linha_selecionada['Horas']) else ""

def limpar_campos_sem_limpar_selecao():
    txt_cpf.value = ""
    txt_nome.value = ""
    txt_dias.value = ""
    txt_horas.value = ""

def limpar_campos(event):
    limpar_campos_sem_limpar_selecao()
    tabela_dados.selection = []

def salvar_registro(event):
    if not txt_cpf.value or not txt_nome.value:
        status_alert.object = "Erro: CPF e Nome são obrigatórios."
        status_alert.alert_type = "danger"
        status_alert.visible = True
        return

    db = SessionLocal()
    try:
        cuidador = db.query(Cuidador).filter_by(cpf_cuidador=txt_cpf.value).first()
        if not cuidador:
            cuidador = Cuidador(cpf_cuidador=txt_cpf.value)
            db.add(cuidador)
            msg = "Cuidador cadastrado com sucesso!"
        else:
            msg = "Cuidador atualizado com sucesso!"

        cuidador.nome_completo = txt_nome.value
        cuidador.dias_disponiveis = txt_dias.value
        cuidador.horario_disponivel = txt_horas.value

        db.commit()
        status_alert.object = msg
        status_alert.alert_type = "success"
        status_alert.visible = True
        carregar_dados()
        limpar_campos(None)
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

def excluir_registro(event):
    selecao = tabela_dados.selection
    if not selecao:
        status_alert.object = "Selecione uma linha na tabela para excluir."
        status_alert.alert_type = "warning"
        status_alert.visible = True
        return

    cpf_selecionado = tabela_dados.value.iloc[selecao[0]]['CPF']
    db = SessionLocal()
    try:
        cuidador = db.query(Cuidador).filter_by(cpf_cuidador=cpf_selecionado).first()
        if cuidador:
            db.delete(cuidador)
            db.commit()
            status_alert.object = "Cuidador excluído com sucesso!"
            status_alert.alert_type = "success"
            status_alert.visible = True
            carregar_dados()
            limpar_campos(None)
    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro ao excluir: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True
    finally:
        db.close()

btn_salvar.on_click(salvar_registro)
btn_excluir.on_click(excluir_registro)
btn_limpar.on_click(limpar_campos)

# Layout moderno em Cards
formulario_card = pn.Card(
    pn.Column(
        pn.Row(txt_cpf, txt_nome),
        pn.Row(txt_dias, txt_horas),
        pn.layout.Divider(),
        pn.Row(btn_salvar, btn_excluir, btn_limpar)
    ),
    title="📝 Formulário de Cuidadores",
    header_css_classes=['premium-card-title']
)

tabela_card = pn.Card(
    tabela_dados,
    title="📋 Base de Cuidadores",
    header_css_classes=['premium-card-title']
)

# Sidebar - Note que o botão "Módulo Cuidadores" agora é o primário e desabilitado (indicando página atual)
sidebar_content = pn.Column(
    pn.pane.Markdown("### 🏥 Menu Principal", margin=(0, 0, 10, 0)),
    pn.widgets.Button(name="Módulo Pacientes", button_type="light", sizing_mode="stretch_width"),
    pn.widgets.Button(name="Módulo Cuidadores", button_type="primary", sizing_mode="stretch_width", disabled=True),
    pn.widgets.Button(name="Módulo Transações", button_type="light", sizing_mode="stretch_width"),
    pn.layout.Divider(),
    pn.pane.Markdown("**Usuário:** Admin\n\n**Status:** Online 🟢", styles={'color': '#cbd5e1', 'font-size': '13px'}),
    sizing_mode="stretch_width"
)

# Criando o Template profissional
template = pn.template.FastListTemplate(
    title='CareLink | Hub de Saúde',
    header_background="#0f172a",
    accent_base_color="#3b82f6",
    sidebar=[sidebar_content],
    main=[
        pn.Column(
            status_alert,
            formulario_card,
            pn.Spacer(height=15),
            tabela_card,
            sizing_mode="stretch_width"
        )
    ]
)

carregar_dados()
template.servable()